# Model to analyze replacement of Russian imports by LNG

- no network conversion used
- analysis of the impact of LNG import capacity increasement

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os

### Import Data

In [2]:
# Specify the path to your Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_path = '\data_base_case_russia.xlsx'
#excel_file_path = '\Data_update_methane_LNG_test.xlsx'

input_file_path  = input_file_path + excel_file_path
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path))


# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(full_input_path, sheet_name='Nodes')
df_commodities = pd.read_excel(full_input_path, sheet_name='Commodities')
df_edges = pd.read_excel(full_input_path, sheet_name='Edges')
df_parameter = pd.read_excel(full_input_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(full_input_path, sheet_name='Supply')

### Create input data structure

In [3]:
# Extract nodes, edges and commodities from the DataFrames
Network_nodes = df_nodes['Nodes'].dropna().tolist()
Commodities = df_commodities['Commodities'].dropna().tolist()
Edges = list(zip(df_edges['Source'], df_edges['Destination']))

# Create a nested dictionary for initial capacities
Initial_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    initial_capacity = row['initial_capacities']

    edge = f"{source}{destination}"

    if commodity not in Initial_capacities:
        Initial_capacities[commodity] = {}

    Initial_capacities[commodity][edge] = initial_capacity

# Create a nested dictionary for max capacities
Max_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    max_capacity = row['max_capacities']

    edge = f"{source}{destination}"

    if commodity not in Max_capacities:
        Max_capacities[commodity] = {}

    Max_capacities[commodity][edge] = max_capacity

# Create a nested dictionary for edge cost
Edge_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    edge_cost = row['costs_edge']

    edge = f"{source}{destination}"

    if commodity not in Edge_cost:
        Edge_cost[commodity] = {}

    Edge_cost[commodity][edge] = edge_cost

# Create a nested dictionary for new pipelines
Pipe_new_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    new_cost = row['new_build_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_new_cost:
        Pipe_new_cost[commodity] = {}

    Pipe_new_cost[commodity][edge] = new_cost

# Create a nested dictionary for pipeline conversion
Pipe_conv_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cost = row['conversion_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_cost:
        Pipe_conv_cost[commodity] = {}

    Pipe_conv_cost[commodity][edge] = conv_cost

# Create a nested dictionary for adjusting the capacity when pipeline conversion
Pipe_conv_factor = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cap_factor = row['conversion_capacity_factor']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_factor:
        Pipe_conv_factor[commodity] = {}

    Pipe_conv_factor[commodity][edge] = conv_cap_factor

# Create a nested dictionary for supply values, skipping 0 and NaN values
Supply_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    supply_node = row['Node']
    supply_value = row['Supply']

    if commodity not in Supply_values:
        Supply_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(supply_value) and supply_value != 0:
        Supply_values[commodity][supply_node] = supply_value

# Create a nested dictionary for node values, skipping 0 and NaN values
Node_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    node_value = row['Supply']

    if commodity not in Node_values:
        Node_values[commodity] = {}

    # Skip 0 and NaN values
    #if not pd.isna(node_value) and node_value != 0:
    if not pd.isna(node_value):
        Node_values[commodity][demand_node] = node_value

In [4]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at infinite (super high) cost
'''

# Create a new dictionary for positive values
positive_values_dict = {}

# Iterate through the outer dictionary
for node, values in Node_values.items():
    # Filter out positive values from the inner dictionary
    positive_values = {key: value for key, value in values.items() if value > 0}
    
    # Check if there are positive values before adding to the new dictionary
    if positive_values:
        positive_values_dict[node] = positive_values

#print("Dictionary with positive values:", positive_values_dict)

# Get all keys from the inner dictionaries
all_keys = [key for values in positive_values_dict.values() for key in values.keys()]

# Remove duplicates to get unique keys
unique_keys = list(set(all_keys))

#print("Unique keys:", unique_keys)

# Create a list with names "shortage_" followed by each key
shortage_list = [f'shortage_{key}' for key in unique_keys]

#print("Shortage list:", shortage_list)

#shortage_edges_list = [(key, shortage) for key, shortage in zip(unique_keys, shortage_list)]
shortage_edges_list = [(key, shortage) for key, shortage in zip(shortage_list, unique_keys)]


#print("Edges list:", shortage_edges_list)

shortage_capacity_dict = {commodity: {f'{key}{shortage}': 1000 for key, shortage in zip(shortage_list, unique_keys)} for commodity in Commodities}
shortage_max_capacity_dict = shortage_capacity_dict
#print("Nested dictionary with capacities:", shortage_capacity_dict)

methane_value = 10000000
hydrogen_value = 10000000

# Your original code
shortage_cost_dict = {
    'Methane': {f'{key}{shortage}': methane_value for key, shortage in zip(shortage_list, unique_keys)},
    'Hydrogen': {f'{key}{shortage}': hydrogen_value for key, shortage in zip(shortage_list, unique_keys)}
}

#shortage_cost_dict = {commodity: {f'{key}{shortage}': -1110 for key, shortage in zip(shortage_list, unique_keys)} for commodity in Commodities}
#print("Shortage cost:", shortage_cost_dict)


In [5]:
'''
Create slack nodes for all supply nodes
enable excess nodes that oversupply is also not a problem
'''

# Create a new dictionary for positive values
positive_values_dict = {}

# Iterate through the outer dictionary
for node, values in Node_values.items():
    # Filter out positive values from the inner dictionary
    positive_values = {key: value for key, value in values.items() if value > 0}
    
    # Check if there are positive values before adding to the new dictionary
    if positive_values:
        positive_values_dict[node] = positive_values

#print("Dictionary with positive values:", positive_values_dict)

# Get all keys from the inner dictionaries
all_keys = [key for values in positive_values_dict.values() for key in values.keys()]

# Remove duplicates to get unique keys
unique_keys = list(set(all_keys))

#print("Unique keys:", unique_keys)

# Create a list with names "excess_" followed by each key
excess_list = [f'{key}_excess' for key in unique_keys]

#print("excess list:", excess_list)

#excess_edges_list = [(key, excess) for key, excess in zip(unique_keys, excess_list)]
excess_edges_list = [(excess, key) for key, excess in zip(excess_list, unique_keys)]


#print("Edges list:", excess_edges_list)

excess_capacity_dict = {commodity: {f'{excess}{key}': 10000 for key, excess in zip(excess_list, unique_keys)} for commodity in Commodities}
excess_max_capacity_dict = excess_capacity_dict
#print("Nested dictionary with capacities:", excess_capacity_dict)

methane_value = 0
hydrogen_value = 10000

# Your original code
excess_cost_dict = {
    'Methane': {f'{excess}{key}': methane_value for key, excess in zip(excess_list, unique_keys)},
    'Hydrogen': {f'{excess}{key}': hydrogen_value for key, excess in zip(excess_list, unique_keys)}
}

#excess_cost_dict = {commodity: {f'{excess}{key}': 0 for key, excess in zip(excess_list, unique_keys)} for commodity in Commodities}
#print("excess cost:", excess_cost_dict)


Print data structure for control

In [6]:
# Print the data for control
#print("Network Nodes:", Network_nodes)
#print("Commodities:", Commodities)
#print("Edges:", Edges)
#print("Initial Capacities:", Initial_capacities)
#print("Max Capacities:", Max_capacities)
#print("Costs per edge Capacities:", Edge_cost)
#print("Costs for new pipelines:", Pipe_new_cost)
#print("Costs for convert pipelines:", Pipe_conv_cost)
#print("Conversion capacity factor:", Pipe_conv_factor)
#print("Node Values:", Node_values)

In [7]:
#check in all nodes are connected via edges

G = nx.Graph()
G.add_edges_from(Edges)

# Check if the graph is connected
if nx.is_connected(G):
    print("The graph is connected.")
else:
    print("The graph is not connected.")

The graph is connected.


In [8]:
#check in all nodes are connected via edges

G = nx.Graph()
G.add_edges_from(Edges)

# Get connected components
connected_components = list(nx.connected_components(G))

# Check if the graph is connected
if len(connected_components) == 1:
    print("The graph is connected.")
else:
    print("The graph is not connected.")
    print("Connected components:")
    for i, component in enumerate(connected_components):
        print(f"Component {i+1}: {component}")

The graph is connected.


In [9]:
# Check if all nodes are connected
if set(Network_nodes) in nx.connected_components(G):
    print("All nodes are connected.")
else:
    print("Not all nodes are connected.")

All nodes are connected.


In [10]:
#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = Pipe_conv_factor

## Model

### Create model

In [11]:
# Create a new model
model = gp.Model("Grid_Transformation")

Set parameter Username
Academic license - for non-commercial use only - expires 2024-12-20


### Define parameters

In [12]:
# Parameters
commodities = Commodities  # Commodity types
#real network elements
network_nodes = Network_nodes # Nodes of the system
network_edges = Edges  # Edges
initial_capacities = Initial_capacities # Initial capacities
max_capacities = Max_capacities  # Maximum capacities
costs_edge = Edge_cost  # Cost to transport from node to node
capacity_new_cost = Pipe_new_cost  # Cost to increase capacity
capacity_change_cost = Pipe_conv_cost  # Cost to increase capacity
node_value = Node_values #contains supply and demand values

#slack parameters shortage
shortage_nodes = shortage_list
shortage_edges = shortage_edges_list
shortage_capacities = shortage_capacity_dict
shortage_cost = shortage_cost_dict

#slack parameters excess
excess_nodes = excess_list
excess_edges = excess_edges_list
excess_capacities = excess_capacity_dict
excess_cost = excess_cost_dict

#complete network of the model
all_edges = network_edges + excess_edges_list + shortage_edges_list

### Define decision variables

In [13]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
x_flow_excess = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    x_flow_excess[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in shortage_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
    for edge in excess_edges:
        x_flow_excess[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_excess_{commodity}_{edge[0]}_{edge[1]}")
model.update()

In [14]:
network_edges_inc_excess = network_edges + excess_edges_list

### Define objective and constraints

In [15]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    #gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    #gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges),
    gp.quicksum(x_flow_shortage[commodity][edge] * shortage_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in shortage_edges)+
    gp.quicksum(x_flow_excess[commodity][edge] * excess_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in excess_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges_inc_excess if edge[0] == node)
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")

for node in excess_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_{commodity}_{node}")

# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in shortage_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][edge], 
                        f"equality_flow_shortage_constraint_{commodity}_{edge}")

#excess node at the sources to avoid infeasible problems.        
for node in excess_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in excess_edges if edge[1] == node) >= 0,
                        f"supply_excess_{commodity}_{node}")


# Add constraint for equality between x_flow and x_flow_excess for the same edge
for commodity in commodities:
    for edge in excess_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_excess[commodity][edge], 
                        f"equality_flow_excess_constraint_{commodity}_{edge}")

#Capacity constraint for excess flow
for commodity in commodities:
    for edge in excess_edges:
        model.addConstr(x_flow_excess[commodity][edge] 
                        <= 100000, 
                        f"capacity_limit_excess{commodity}_{edge[0]}_{edge[1]}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")


# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[Commodities[0]][edge] + Change[Commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[Commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [16]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1977 rows, 1912 columns and 3940 nonzeros
Model fingerprint: 0xa6ddb549
Variable types: 1474 continuous, 438 integer (438 binary)
Coefficient statistics:
  Matrix range     [8e-01, 1e+04]
  Objective range  [1e+00, 1e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-02, 1e+05]
Presolve removed 1923 rows and 1743 columns
Presolve time: 0.02s
Presolved: 54 rows, 169 columns, 275 nonzeros
Variable types: 169 continuous, 0 integer (0 binary)

Root relaxation: objective 2.196882e+06, 101 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    219688

### Results processing

In [17]:
# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
else:
    print("No optimal solution found.")

Optimal solution found!


In [18]:
# Assuming commodities, edges, x_flow, y_new_cap, Change, and z_conv_cap are defined in your code

# Create lists to store the data
results_data = []
columns = ["Commodity", "Edge", "Flow", "New Capacity", "Switched", "Changed Capacity"]
excess_data = []
excess_columns = ["Commodity", "Edge", "Flow"]
shortage_data = []
shortage_columns = ["Commodity", "Edge", "Flow"]

# Check if the model has an optimal solution
if model.status == GRB.OPTIMAL:
    for commodity in commodities:
        for edge in network_edges:
            # Append data to the list
            results_data.append([commodity,
                                 edge, 
                                 x_flow[commodity][edge].x, 
                                 y_new_cap[commodity][edge].x, 
                                 Change[commodity][edge].x, 
                                 z_conv_cap[commodity][edge].x])
        for edge in excess_edges:    
            excess_data.append([commodity, 
                              edge, 
                              x_flow_excess[commodity][edge].x])
        for edge in shortage_edges:    
            shortage_data.append([commodity, 
                              edge, 
                              x_flow_shortage[commodity][edge].x])
else:
    print("No optimal solution found.")

if model.status == GRB.OPTIMAL:
    # Create a DataFrame
    results_df = pd.DataFrame(results_data, columns=columns)
    excess_df = pd.DataFrame(excess_data, columns=excess_columns)
    shortage_df = pd.DataFrame(shortage_data, columns=shortage_columns)

In [19]:
#model.write()

In [20]:
methane_rows_df = results_df[results_df['Commodity'].str.contains('Methane', case=False)]
methane_rows_df = methane_rows_df.drop(columns=['New Capacity', 'Switched', 'Changed Capacity'])
methane_rows_df

,Commodity,Edge,Flow
0,Methane,"(BE_LNG, BE)",111.378000
1,Methane,"(FR_LNG, FR)",322.410000
2,Methane,"(GR_LNG, GR)",73.275000
3,Methane,"(IT_LNG, IT)",155.831500
4,Methane,"(HR_LNG, HR)",25.402000
...,...,...,...
134,Methane,"(RS_Prod, RS)",3.995930
135,Methane,"(SK_Prod, SK)",0.651669
136,Methane,"(SI_Prod, SI)",0.052113
137,Methane,"(TR_Prod, TR)",4.274473


In [21]:
search_element = 'ES'

# Filter rows based on whether the search element is present in 'Column1'
string_df = results_df[results_df['Edge'].astype(str).str.contains(search_element, case=False, na=False)]

# Display the resulting DataFrame
string_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity
9,Methane,"(ES_LNG, ES)",398.454,655.56700,0.0,0.00000
59,Methane,"(FR, ES)",0.000,60.06878,0.0,0.00000
89,Methane,"(ES, FR)",81.906,81.90600,0.0,0.00000
90,Methane,"(ES, PT)",0.000,52.56000,0.0,0.00000
91,Methane,"(PT, ES)",0.000,29.20000,0.0,0.00000
103,Methane,"(MA, ES)",0.000,161.62200,0.0,0.00000
111,Methane,"(DZ, ES)",0.000,123.04150,0.0,0.00000
138,Methane,"(ES_Prod, ES)",0.000,10000.00000,0.0,0.00000
148,Hydrogen,"(ES_LNG, ES)",0.000,9344.43300,1.0,655.56700
198,Hydrogen,"(FR, ES)",0.000,9939.93122,1.0,60.06878


In [22]:
excess_df

methane_excess_df = excess_df[excess_df['Commodity'].str.contains('Methane', case=False)]
methane_excess_df

,Commodity,Edge,Flow
0,Methane,"(BE_LNG, BE_LNG_excess)",0.000000
1,Methane,"(BE_Prod, BE_Prod_excess)",0.000000
2,Methane,"(NL_Prod, NL_Prod_excess)",0.000000
3,Methane,"(UK_Prod, UK_Prod_excess)",0.000000
4,Methane,"(PL_Prod, PL_Prod_excess)",0.000000
5,Methane,"(GR_LNG, GR_LNG_excess)",0.000000
6,Methane,"(DE_Prod, DE_Prod_excess)",0.000000
7,Methane,"(LT_LNG, LT_LNG_excess)",0.000000
8,Methane,"(IT_LNG, IT_LNG_excess)",0.000000
9,Methane,"(DZ_Prod, DZ_Prod_excess)",393.608000


In [23]:
shortage_df

methane_shortage_df = shortage_df[shortage_df['Commodity'].str.contains('Methane', case=False)]
methane_shortage_df

,Commodity,Edge,Flow
0,Methane,"(shortage_BE_LNG, BE_LNG)",0.0
1,Methane,"(shortage_BE_Prod, BE_Prod)",0.0
2,Methane,"(shortage_NL_Prod, NL_Prod)",0.0
3,Methane,"(shortage_UK_Prod, UK_Prod)",0.0
4,Methane,"(shortage_PL_Prod, PL_Prod)",0.0
5,Methane,"(shortage_GR_LNG, GR_LNG)",0.0
6,Methane,"(shortage_DE_Prod, DE_Prod)",0.0
7,Methane,"(shortage_LT_LNG, LT_LNG)",0.0
8,Methane,"(shortage_IT_LNG, IT_LNG)",0.0
9,Methane,"(shortage_DZ_Prod, DZ_Prod)",0.0


In [24]:
methane_rows_df.to_excel("output1.xlsx")